# Module 03: OOP for ML Projects — Solutions

Complete solutions to all exercises.

## Exercise 1: Simple Model Wrapper

In [ ]:
class SimpleModel:
    """A simple model wrapper with train/predict interface."""
    
    def __init__(self, model_type="regression"):
        """Initialize model with type."""
        self.model_type = model_type
        self.coefficients = None
        self._is_trained = False
    
    def train(self, X, y):
        """Train the model on data."""
        print("Training", self.model_type, "model on", len(X), "samples")
        self.coefficients = [0.5] * len(X[0])
        self._is_trained = True
        return self
    
    def predict(self, X):
        """Make predictions. Raises error if not trained."""
        if not self._is_trained:
            raise ValueError("Model not trained yet. Call train() first.")
        return [0.0] * len(X)


# Test
model = SimpleModel("regression")
model.train([[1, 2], [3, 4]], [1.5, 3.5])
print(model.predict([[5, 6]]))

untrained = SimpleModel()
try:
    untrained.predict([[1, 2]])
except ValueError as e:
    print("Error caught:", e)

## Exercise 2: Inheritance Hierarchy for Models

In [ ]:
from abc import ABC, abstractmethod

class BaseModel(ABC):
    """Abstract base for all models."""
    
    @abstractmethod
    def fit(self, X, y):
        pass
    
    @abstractmethod
    def predict(self, X):
        pass


class LinearModel(BaseModel):
    """Linear model with learned weights."""
    
    def __init__(self):
        self.weights = None
    
    def fit(self, X, y):
        n_features = len(X[0])
        self.weights = [0.0] * n_features
        for j in range(n_features):
            values = [X[i][j] for i in range(len(X))]
            targets = [y[i] for i in range(len(y))]
            mean_x = sum(values) / len(values)
            mean_y = sum(targets) / len(targets)
            num = sum((values[i] - mean_x) * (targets[i] - mean_y) for i in range(len(values)))
            den = sum((values[i] - mean_x) ** 2 for i in range(len(values)))
            self.weights[j] = num / den if den != 0 else 0.0
        print("LinearModel weights:", self.weights)
    
    def predict(self, X):
        preds = []
        for x in X:
            preds.append(sum(w * x[j] for j, w in enumerate(self.weights)))
        return preds


class KNNModel(BaseModel):
    """Simple k-nearest neighbors classifier."""
    
    def __init__(self, k=1):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        print("KNNModel stored", len(X), "training samples")
    
    def predict(self, X):
        preds = []
        for x in X:
            distances = []
            for i, x_train in enumerate(self.X_train):
                dist = sum((x[j] - x_train[j]) ** 2 for j in range(len(x)))
                distances.append((dist, self.y_train[i]))
            distances.sort(key=lambda p: p[0])
            nearest_labels = [distances[i][1] for i in range(self.k)]
            preds.append(max(set(nearest_labels), key=nearest_labels.count))
        return preds


# Test
models = [LinearModel(), KNNModel(k=3)]
X = [[1, 2], [3, 4], [5, 6]]
y = [0, 1, 0]
for m in models:
    m.fit(X, y)
    print(type(m).__name__, "predicts:", m.predict([[2, 3]]))

## Exercise 3: Dataset Class with Properties

In [ ]:
class MLDataset:
    """Dataset wrapper with properties and split functionality."""
    
    def __init__(self, features, labels=None, feature_names=None):
        self._features = features
        self._labels = labels
        self._feature_names = feature_names
    
    @property
    def num_samples(self):
        return len(self._features)
    
    @property
    def num_features(self):
        if not self._features:
            return 0
        return len(self._features[0])
    
    @property
    def shape(self):
        return (self.num_samples, self.num_features)
    
    @property
    def feature_names(self):
        if self._feature_names:
            return self._feature_names
        return ["f" + str(i) for i in range(self.num_features)]
    
    def train_test_split(self, test_size=0.2, random_state=None):
        n = len(self._features)
        indices = list(range(n))
        if random_state is not None:
            random_seed = random_state
            shuffled = []
            remaining = indices[:]
            while remaining:
                idx = (random_seed * 7 + 13) % len(remaining)
                shuffled.append(remaining.pop(idx))
                random_seed = random_seed + 1
            indices = shuffled
        split = int(n * (1 - test_size))
        train_indices = indices[:split]
        test_indices = indices[split:]
        
        train_features = [self._features[i] for i in train_indices]
        test_features = [self._features[i] for i in test_indices]
        train_labels = [self._labels[i] for i in train_indices] if self._labels else None
        test_labels = [self._labels[i] for i in test_indices] if self._labels else None
        
        return (
            MLDataset(train_features, train_labels, self._feature_names),
            MLDataset(test_features, test_labels, self._feature_names)
        )
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        if self._labels:
            return (self._features[idx], self._labels[idx])
        return self._features[idx]
    
    def __str__(self):
        return "MLDataset(" + str(self.num_samples) + " samples, " + str(self.num_features) + " features)"


# Test
data = MLDataset([[1, 2], [3, 4], [5, 6], [7, 8]], [0, 1, 0, 1], feature_names=["x1", "x2"])
print(data)
print("Shape:", data.shape)
print("First sample:", data[0])
train, test = data.train_test_split(test_size=0.25, random_state=42)
print("Train size:", len(train), "Test size:", len(test))

## Exercise 4: Model Registry with @classmethod

In [ ]:
class ModelRegistry:
    """Class-level registry for tracking models."""
    
    _registry = {}
    
    @classmethod
    def register(cls, name, model):
        """Register a model by name."""
        cls._registry[name] = model
        print("Registered model:", name)
    
    @classmethod
    def get(cls, name):
        """Get a registered model by name."""
        if name not in cls._registry:
            raise KeyError("Model not found: " + name)
        return cls._registry[name]
    
    @classmethod
    def list_models(cls):
        """List all registered model names."""
        return list(cls._registry.keys())
    
    @classmethod
    def run_all(cls, X, y):
        """Train all registered models."""
        results = {}
        for name, model in cls._registry.items():
            print("Training:", name)
            model.fit(X, y)
            results[name] = model.predict(X)
        return results
    
    @classmethod
    def clear(cls):
        """Clear the registry."""
        cls._registry = {}


# Test
class DummyModel:
    def __init__(self, name):
        self.name = name
    def fit(self, X, y):
        print(self.name, "trained")
    def predict(self, X):
        return [0]

ModelRegistry.register("model_a", DummyModel("A"))
ModelRegistry.register("model_b", DummyModel("B"))
print("Available:", ModelRegistry.list_models())
ModelRegistry.run_all([[1, 2]], [0])
ModelRegistry.clear()

## Exercise 5: Full ML Pipeline Component

In [ ]:
from abc import ABC, abstractmethod

class BaseTransformer(ABC):
    """Abstract base for all transformers."""
    
    @abstractmethod
    def fit(self, X):
        pass
    
    @abstractmethod
    def transform(self, X):
        pass
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


class IdentityEncoder(BaseTransformer):
    """Pass-through transformer."""
    
    def fit(self, X):
        print("IdentityEncoder: no parameters to learn")
        return self
    
    def transform(self, X):
        print("IdentityEncoder: passing through")
        return X


class OneHotEncoder(BaseTransformer):
    """Simple one-hot encoder for the first feature."""
    
    def __init__(self):
        self.categories_ = None
    
    def fit(self, X):
        categories = []
        for x in X:
            if x[0] not in categories:
                categories.append(x[0])
        self.categories_ = sorted(categories)
        print("OneHotEncoder found categories:", self.categories_)
        return self
    
    def transform(self, X):
        result = []
        for x in X:
            encoded = [1.0 if x[0] == cat else 0.0 for cat in self.categories_]
            row = encoded + list(x[1:])
            result.append(row)
        return result


class MLPipeline:
    """Pipeline with transformers and a final model."""
    
    def __init__(self, transformers, model):
        self.transformers = transformers
        self.model = model
    
    def fit(self, X, y):
        X_current = X
        for t in self.transformers:
            t.fit(X_current)
            X_current = t.transform(X_current)
        self.model.train(X_current, y)
        return self
    
    def predict(self, X):
        X_current = X
        for t in self.transformers:
            X_current = t.transform(X_current)
        return self.model.predict(X_current)


# Test
pipeline = MLPipeline(
    transformers=[IdentityEncoder()],
    model=SimpleModel("regression")
)
X = [[1.0, 2.0], [3.0, 4.0]]
y = [5.0, 7.0]
pipeline.fit(X, y)
print(pipeline.predict([[2.0, 3.0]]))

## Exercise 6: Hyperparameter Grid with @dataclass

In [ ]:
from dataclasses import dataclass
from itertools import product


@dataclass
class HyperparameterConfig:
    """Single hyperparameter configuration."""
    learning_rate: float
    batch_size: int
    num_layers: int


class HyperparameterGrid:
    """Grid of hyperparameter configurations."""
    
    def __init__(self, learning_rates, batch_sizes, num_layers):
        self.configs = []
        for lr, bs, nl in product(learning_rates, batch_sizes, num_layers):
            self.configs.append(HyperparameterConfig(lr, bs, nl))
    
    @classmethod
    def from_dict(cls, param_dict):
        """Create grid from a dictionary of parameter lists."""
        return cls(
            learning_rates=param_dict.get("learning_rate", [0.01]),
            batch_sizes=param_dict.get("batch_size", [32]),
            num_layers=param_dict.get("num_layers", [2])
        )
    
    def __len__(self):
        return len(self.configs)
    
    def __getitem__(self, idx):
        return self.configs[idx]
    
    def __str__(self):
        return "HyperparameterGrid(" + str(len(self)) + " configs)"
    
    def __iter__(self):
        return iter(self.configs)


# Test
grid = HyperparameterGrid(
    learning_rates=[0.001, 0.01],
    batch_sizes=[16, 32],
    num_layers=[2, 3]
)
print("Total configs:", len(grid))
for i in range(len(grid)):
    print(grid[i])

# Test from_dict
grid2 = HyperparameterGrid.from_dict({
    "learning_rate": [0.1, 0.01],
    "batch_size": [64]
})
print("\nFrom dict - total configs:", len(grid2))
for c in grid2:
    print(c)